# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = next(
    (
        parent / 'data'
        for parent in (Path.cwd(), *Path.cwd().parents)
        if all((parent / 'data' / name).is_file() for name in ('job_snippets.jsonl', 'golden_set.jsonl'))
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(f'Could not locate the project data files from {Path.cwd()}')

In [3]:
DATA_DIR

WindowsPath('c:/Users/gayat/Downloads/Week 5_Graded Mini Project Documents/AI-RAG_MP1_PromptLab/MP1_PromptLab/data')

In [4]:
snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]

In [5]:
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [6]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    # TODO: return a messages list like [{'role': 'user', 'content': '...'}]
    prompt = f"""Analyse the job snippets along with the job id and provide the output in the json format having details id(which is the job_id from the snippet), company, role, years_experience_required and notes.
Make sure you provide the exact values.
If years of experience is not mentioned, just mention as 0, not None.
f"Job posting:
{snippet_text}
"""

    return [{"role": "user", "content": prompt}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    prompt = f"""Analyse the job snippets along with the job id and privde the output in the json format having details the detail job_id(which is already provided), company_name, role, number of years and notes.    
Make sure you copy the exact job ID being provided.
If years of experience is not mentioned, just mention as 0, not None.
Example 1 Input:{{"id": "j01", "snippet": "Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems."}}
Example 1 Output: {{"id": "j01", "company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5, "notes": "Looking for backend development experience and strong skills in Python and distributed systems."}}
    
Example 2 Input:{{"id": "j02", "snippet": "We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."}}
Example 2 Output: {{"id": "j02", "company": "Northwind Ltd", "role": "Data Analyst", "years_experience_required": 2, "notes": "clean — preferred used but still a stated minimum"}}

Job posting:
{snippet_text}
"""

    return [{"role": "user", "content": prompt}]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    # TODO: 'You are an expert recruiter... Output JSON with these exact fields...'
    # raise NotImplementedError
    prompt=f"""You are an expert recruiter. Extract the company, role, and required years of experience.
Return only JSON with these fields: id, company, role, years_experience_required and notes.
Use null if years of experience are not stated.
Make sure you copy the exact job ID being provided.
If years of experience is not mentioned, just mention as 0, not None.

Example input: {{"id": "j01", "snippet": "Acme Corp is hiring a Senior Software Engineer with 5+ years of experience."}}
Example output: {{"id":"j01","company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5,notes:"clean — clear company + role + years"}}

Job posting:
{snippet_text}
"""

    return [{"role": "user", "content": prompt}]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — ask the model to analyse before extracting."""
    prompt = f"""
    Analyze this job posting carefully. Identify the job id,company, job title, explicitly stated years of experience and notes.
Include the additional details in the notes. 
Do not infer missing information; use null if years are not specified. 
Treat '5+' as 5 and a range like '3 to 5 years' as 3. 
Make sure you copy the exact job ID being provided.
If years of experience is not mentioned, just mention as 0, not None.
Then return the result as valid JSON using exactly these keys: id, company, role, years_experience_required and notes(include the additional details from the snippet).

Follow this example:
Example input:
{{"id":"j02","snippet":"Northwind Ltd. is hiring a Data Analyst. Two years minimum experience preferred. You'll use SQL and build dashboards."}}

Example output:
{{"id":"j02","company":"Northwind Ltd.","role":"Data Analyst","years_experience_required":2,"notes":"Uses SQL and builds dashboards."}}
Job posting:
{snippet_text}
"""

    return [{"role": "user", "content": prompt}]


STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

In [7]:
for name, prompt_fn in STRATEGIES.items():
    prompt_fn(snippets[0]["snippet"])
    print(f"{name}: prompt built")

zero_shot: prompt built
few_shot: prompt built
structured: prompt built
cot: prompt built


## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [8]:
import re

def parse_response(text: str) -> dict | None:
    text = re.sub(
        r"\A```(?:json)?\s*|\s*```\Z",
        "",
        text.strip(),
        flags=re.IGNORECASE,
    ).strip()

    # Find a JSON object even if the model adds explanatory text before it.
    decoder = json.JSONDecoder()
    for start, character in enumerate(text):
        if character == "{":
            try:
                result, _ = decoder.raw_decode(text[start:])
            except json.JSONDecodeError:
                continue
            if isinstance(result, dict):
                return result

    return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet and capture its output and metrics."""
    # messages = STRATEGIES[strategy_name](snippet["snippet"])
    messages = STRATEGIES[strategy_name](snippet)
   
    started = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency = time.perf_counter() - started

    if not response.choices or response.choices[0].message.content is None:
        raise RuntimeError(f"No text returned for {strategy_name}, {snippet['id']}.")
    if response.usage is None:
        raise RuntimeError(f"Token usage missing for {strategy_name}, {snippet['id']}.")

    raw_response = response.choices[0].message.content
    usage = response.usage
    cost = (
        usage.prompt_tokens * RATES[MODEL]["in"]
        + usage.completion_tokens * RATES[MODEL]["out"]
    )

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw_response": raw_response,
        "parsed_extraction": parse_response(raw_response),
        "cost": cost,
        "latency": latency,
    }


async def run_all() -> list[dict]:
    """Run every strategy against every snippet concurrently."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return await asyncio.gather(*tasks)

In [9]:
# Run it

results = await run_all()

In [10]:
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "id": "j01",\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5,\n  "notes": "Strong skills in Python and distributed systems."\n}\n```',
 'parsed_extraction': {'id': 'j01',
  'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5,
  'notes': 'Strong skills in Python and distributed systems.'},
 'cost': 5.175e-05,
 'latency': 4.425926099997014}

In [11]:
for result in results:
    print(result)

{'strategy': 'zero_shot', 'snippet_id': 'j01', 'raw_response': '```json\n{\n  "id": "j01",\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5,\n  "notes": "Strong skills in Python and distributed systems."\n}\n```', 'parsed_extraction': {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'Strong skills in Python and distributed systems.'}, 'cost': 5.175e-05, 'latency': 4.425926099997014}
{'strategy': 'zero_shot', 'snippet_id': 'j02', 'raw_response': '```json\n{\n  "id": "j02",\n  "company": "Northwind Ltd.",\n  "role": "Data Analyst",\n  "years_experience_required": "2",\n  "notes": "2 years minimum experience preferred."\n}\n```', 'parsed_extraction': {'id': 'j02', 'company': 'Northwind Ltd.', 'role': 'Data Analyst', 'years_experience_required': '2', 'notes': '2 years minimum experience preferred.'}, 'cost': 4.9949999999999994e-05, 'latency': 4.104819900006987}
{'strategy': 

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [12]:
import re

FIELDS = ['company', 'role', 'years_experience_required']

def _normalise(value):
    if value is None:
        return None
    text = ' '.join(str(value).strip().casefold().split())
    if text in {'', 'null', 'none', 'not specified', 'n/a'}:
        return None
    return text

In [13]:
def _normalise_years(value):
    text = _normalise(value)
    if text is None:
        return None
    number_words = {'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10}
    match = re.search(r'\d+', text)
    if match:
        return int(match.group())
    for word, number in number_words.items():
        if re.search(rf'\b{word}\b', text):
            return number
    if 'no prior experience' in text or 'no experience required' in text:
        return 0
    return text

In [14]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Count matches for company, role, and years (5+ and 5 years both count as 5). Return 0-3."""
    if not isinstance(extracted, dict):
        return 0
    matches = 0
    for field in FIELDS:
        normalise = _normalise_years if field == 'years_experience_required' else _normalise
        matches += normalise(extracted.get(field)) == normalise(gold.get(field))
    return matches

In [15]:
async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Ask GPT-4o to rate extraction quality from 1 to 4; return the integer score."""
    rubric = {
        '4': 'All three target fields are correct and supported by the job posting.',
        '3': 'Two of three target fields are correct; no material unsupported claim.',
        '2': 'One field is correct, or there is a material unsupported/inaccurate value.',
        '1': 'No fields are correct, or the extraction is missing/unusable.'
    }
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': 'You are a strict evaluator of information extraction. Compare only company, role, and years_experience_required. Use the job posting and reference answer. Treat null as correct only when no years requirement is stated. Return JSON only as {"score": 1}, with an integer score from 1 to 4.'},
            {'role': 'user', 'content': json.dumps({'job_posting': snippet_text, 'reference': {k: gold.get(k) for k in FIELDS}, 'extraction': extracted if isinstance(extracted, dict) else None, 'rubric': rubric}, ensure_ascii=False)}
        ],
    )
    payload = json.loads(response.choices[0].message.content)
    score = int(payload['score'])
    if score not in (1, 2, 3, 4):
        raise ValueError(f'Judge returned invalid score: {score}')
    return score

In [16]:
# Apply scores to every strategy/snippet result.
snippet_by_id = {row['id']: row for row in snippets}
judge_limit = asyncio.Semaphore(5)  # avoid sending all judge calls at once

async def score_one(result):
    snippet = snippet_by_id[result['snippet_id']]
    gold = golden[result['snippet_id']]
    extracted = result['parsed_extraction']
    async with judge_limit:
        judge_score = await score_llm_judge(snippet['snippet'], extracted, gold)
    return {
        **result,
        'accuracy': score_accuracy(extracted, gold),
        'parse_success': isinstance(extracted, dict),
        'llm_judge_score': judge_score,
        'cost_usd': result['cost'],
        'latency_s': result['latency'],
    }

scored = await asyncio.gather(*(score_one(result) for result in results))
print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [17]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.9,0.001,3.786
few_shot,2.8,1.0,3.8,0.001,3.982
structured,2.8,1.0,3.9,0.001,3.732
zero_shot,2.7,1.0,3.9,0.001,4.235


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```